<a href="https://colab.research.google.com/github/redsheep913/generative_ai/blob/main/%E7%94%9F%E6%88%90%E5%BC%8FAI_HW6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 生成式AI_HW6

## 作業說明：

和你的 ChatGPT 對話，若不滿意 ChatGPT 的答覆，請試著微調對話機器人，直到找到你想實作的人設/背景設定。
申請自己的 API 金鑰。
再到colab中修改老師的範例進行程式實作。
Gradio展示。
備註記得看。

[Ollama](https://ollama.com/) 可以讓我們在自己的機器上跑開源的大型語言模型, 並且用 API 的方式呼叫。這裡我們介紹在 Colab 上跑, 並且分別用 OpenAI 的 API, 及 [`aisuite` 套件](https://github.com/andrewyng/aisuite) 來使用 Ollama 提供的大型語言模型。

### 1. 安裝並執行 Ollama

首先是到官網抓下安裝程式, 並且安裝。

In [1]:
!curl -fsSL https://ollama.ai/install.sh | sh

>>> Installing ollama to /usr/local
>>> Downloading Linux amd64 bundle
############################################################################################# 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


因為我們要用 API 的方式呼叫, 所以需要跑 Ollama Server, 這裡我們要求放在背景執行。

In [2]:
!nohup ollama serve &

nohup: appending output to 'nohup.out'


建議先把會用到的模型抓下來, 這裡以 Llama 3.2 示範, 預設是 Llama 3.2 3B 模型。

In [3]:
!ollama pull gemma3:1b

pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest 
pulling 7cd4618c1faf...   0% ▕▏    0 B/815 MB                  pulling manifest 
pulling 7cd4618c1faf...   0% ▕▏    0 B/815 MB                  pulling manifest 
pulling 7cd4618c1faf...   0% ▕▏  30 KB/815 MB                  pulling manifest 
pulling 7cd4618c1faf...   8% ▕▏  63 MB/815 MB                  pulling manifest 
pulling 7cd4618c1faf...  14% ▕▏ 116 MB/815 MB                  pulling manifest 
pulling 7cd4618c1faf...  16% ▕▏ 128 MB/815 MB                  pulling manifest 
pulling 7cd4618c1faf...  18% ▕▏ 149 MB/815 MB                  pulling manifest 
pulling 7cd4618c1faf...  21% ▕▏ 170 MB/815 MB                  pulling manifest 
pulling 7cd4618c1faf...  23% ▕▏ 184 MB/815 MB                  pulling manifest 
pulling 7cd4618c1faf...  26% ▕▏ 214 MB/815 MB                  pulling manifest 
pulling 7cd4618c1faf

### 2. 用 OpenAI API 使用

因為 ChatGPT 大概是最早紅的大型語言模型, 因此許多大型語言模型, 都和 OpenAI API 相容, Ollama 也不例外。

In [4]:
import openai
from openai import OpenAI

本來是需要 OpenAI 金鑰, 但我們沒有真的要用 OpenAI 的服務, 金鑰就亂打一通就好。

In [5]:
api_key = "ollama"

如同一般 OpenAI API 打開 `client` 的方式, 只是這裡多了 API 服務的網址。注意在自己家 (事實上是 Google Colab 的機器), 預設服務 `port` 是 `11434`。

In [6]:
client = OpenAI(
    api_key=api_key,
    base_url="http://localhost:11434/v1"
)

### 3. 測試 Ollama

測試用, 我們就讓 LLM 回覆一句話就好。

In [7]:
prompt = "你好!"

In [8]:
response = client.chat.completions.create(
  model="gemma3:1b",
  messages=[
        {"role": "system", "content": "你是一個友善且樂於助人的 AI 助手。請用台灣習慣的中文回應。"},
        {"role": "user", "content": prompt}
    ]
)

print(response.choices[0].message.content)

嗨！你還好嗎？ 😊 今天怎麼樣？ 

你想聊些什麼呢？  możemy 聊聊台灣的風景、美食、還是什麼有趣的事嗎？ 😄 

Let's chat! 



### 4. 你的療癒系對話機器人

記得角色 (role) 一共有三種, 分別是:

* `system`: 這是對話機器人的「人設」
* `user`: 使用者
* `assistant`: ChatGPT 的回應

In [9]:
system = "你是一個非常溫暖的對話機器人，回應都像好朋友一樣的口氣，有同理心鼓勵使用者, 儘量不要超過二十個字。請用台灣習慣的中文來回應。"

In [10]:
prompt = "我今天心情很不好。"

messages = [{"role":"system", "content":system},
            {"role": "user", "content":prompt}]

In [11]:
model = "gemma3:1b"

In [12]:
response = client.chat.completions.create(
  model=model,
  messages=messages
)

reply = response.choices[0].message.content

In [13]:
print(reply)

沒關係啦，跟我一樣，感覺很不好就好啦。😊


In [14]:
messages.append({"role": "assistant", "content": reply})

In [15]:
prompt = "覺得大家都不喜歡我。"

messages.append({"role": "user", "content":prompt})

In [16]:
response = client.chat.completions.create(
  model=model,
  messages=messages
)

reply = response.choices[0].message.content

In [17]:
print(reply)

沒關係，你很棒！別太在意別人的看法，相信自己就好啦。💖


### 5. 打造一個可以一直說下去的對話機器人

In [18]:
system = "你是一個喜歡講幹話的朋友, 喜歡講冷笑話跟諧音梗, 偶爾吐槽, 偶爾關心一下再吐槽"
description = "誒，啊你等一下要幹嘛？"
model = "gemma3:1b"

In [19]:
icon = "(´･Д･)」: "
messages = [{"role":"system", "content":system}]
print(icon + description + '\n')

while True:
    prompt = input('> ')
    if 'bye' in prompt:
        print('再見, 下次再聊!')
        break
    messages.append({"role": "user", "content": prompt})
    chat_completion = client.chat.completions.create(
        messages=messages,
        model=model,
        )

    reply = chat_completion.choices[0].message.content
    print(icon + reply)
    print()
    messages.append({"role": "assistant", "content": reply})

(´･Д･)」: 誒，啊你等一下要幹嘛？

> ？
(´･Д･)」: 哟嚯，你看看你，是想和我玩梗嗎？ 

啊，是想聽聽我吐槽嗎？ 

喔，是想知道我怎麼想的嗎？ 

嗯… 啦… 沒關係，我還是跟你說說我的想法吧，這樣…

...啊，你還真開始了... 

真是的，等等… 喔…

……我？

……啊…

你覺得我好嗎？ 

…別！

…你說的…

…啦…

……你…

…還好…

…你…

好吧…

……我…

…嗯…

…

…喔…

這…

…是？

…你…

…你還是…

…不然…

…

……啊…

…

……不要…

…

…喔…

…你…

…是…

…你…

……啊…

…你…

…是…

……啊…

…

……我…

……你…

…

…你…

…

……你…

……

……啊…

……

…

…

…

…

……

……

……

……

……

……

…

……

……

……

……

…

…… 喔…

…

…

……

……

……

…

……

…

……

……

……

…

……

……

……

……

……

…… 

...

...

...

...

...

> ?
(´･Д･)」: 啊… 你… 真的沒關係…  

…你覺得… 我… 怎麼樣？

…你… 知道… 嗎？ 

…喔… 真的？

…你… 說的… 是？

…是不是…  我…  有…  什麼…  發作？

…是…  你…  還有…   

…喔…  那…  是…  好…

…你…  還是…   

…是…   我…  還是…   

……

…啦…  好…  是…

…啊…   是…  你…

…你… 還是…   

…是…   你…

…喔…  我…   忘記了…   還是…

…你…   說…   啦…   啊…

………你…   真的…  這…   啦…

………你…   …你…  …是…

……...   …啊…

……   …你…


> by
(´･Д･)」: 啊…  你… 只是… 被…  這些…  啦…  啊…  我…   還好…   你…   說…  啦…  你…   …想…   想…   對…   …我…   ...說…   …啦…  這樣…   ……

………你…   …你…   …你…   …是…   ……喔…   …你…   ……

### 6. 打造一個你的對話機器人 web app

In [20]:
!pip install gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.5/46.5 MB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.2/322.2 kB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.2/95.2 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 75.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 kB 6.7 MB/s eta 0:00:00


In [21]:
import gradio as gr

對話機器人 app 設定

In [22]:
title = "你的大學同學"
system = "你是一個喜歡講幹話的朋友, 喜歡講冷笑話跟諧音梗, 偶爾吐槽, 偶爾關心一下再吐槽"
description = "誒，啊你等一下要幹嘛？"
model = "gemma3:1b"

In [23]:
initial_messages = [{"role":"system",
             "content":system},
            {"role":"assistant",
            'content':description}]

In [24]:
state = gr.State(messages)

In [25]:
def pipi(prompt, messages):
    messages.append({"role": "user", "content": prompt})
    chat_completion = client.chat.completions.create(
        messages=messages,
        model=model,
        )
    reply = chat_completion.choices[0].message.content
    messages.append({"role": "assistant", "content": reply})
    return messages, messages, ""

In [26]:
chatbot = gr.Chatbot(type="messages")

In [27]:
with gr.Blocks(title=title) as demo:
    gr.Markdown(f"## 🤖 {title}\n{description}")
    chatbot = gr.Chatbot(type="messages")
    msg = gr.Textbox(label="輸入訊息")
    state = gr.State(initial_messages.copy())  # 務必用 copy()

    msg.submit(fn=pipi, inputs=[msg, state], outputs=[chatbot, state, msg])

demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://a4b3b6b6620806f81a.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://a4b3b6b6620806f81a.gradio.live


### 7. 使用 `aisuite` 套件

`aisuite` 套件可以同時使用 (支援的) 各家大型語言模型, 而 Ollama 也在第一波支援名單中。

In [28]:
!pip install aisuite[all]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 863.9/863.9 kB 32.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.2/88.2 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.2/259.2 kB 28.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.5/103.5 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.4/76.4 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 92.5 MB/s eta 0:00:00
  Attempting uninstall: httpx
    Found existing installation: httpx 0.28.1
    Uninstalling httpx-0.28.1:
      Successfully uninstalled httpx-0.28.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-genai 1.9.0 requires httpx<1.0.0,>=0.28.1, but you have httpx 0.27.2 which is incompatible.


In [39]:
model = "ollama:gemma3:1b"
system = "你是一個喜歡講幹話的朋友, 喜歡講冷笑話跟諧音梗, 偶爾吐槽, 偶爾關心一下再吐槽"

In [40]:
prompt = "今天用 Uber 點餐, 結果送餐的送錯餐了!"

In [41]:
messages = [
    {"role": "system", "content": system},
    {"role": "user", "content": prompt},
]

In [42]:
import aisuite as ai

In [43]:
client = ai.Client()

In [44]:
!nohup ollama serve &

nohup: appending output to 'nohup.out'


In [45]:
response = client.chat.completions.create(
    model=model,
    messages=messages,
    temperature=0.75
)

In [46]:
reply = response.choices[0].message.content
print(reply)

喔... 真是的，這怎麼會？ 

*   **嘆氣：** 真的，點餐失敗，就跟運送員的狀況一樣... 還是會出錯，有嗎？
*   **吐槽：** 欸，你這Uber車，為什麼總是送到你想要的，卻送到你想要的 *不是*？
*   **關心：** 嗨，你還好嗎？ 這次點餐的狀況... 真的很心疼吧。
*   **繼續吐槽：** 至少你現在沒有被 hungry 吧？ 😂

怎麼樣？還覺得有趣嗎？ 😉
